## Context

- BM25 and Contriever showed different results in the evaluation.
- BM25 performed better in bridge questions, since it performs better at finding rarer terms.
- On the other hand, the Contriever embeddings were better at answering comparison questions given that it takes into account the "meaning" of the text, since it uses self-attention
- Here we explore how to possibly combine both methods and evaluate if the combination performs better.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from collections import defaultdict

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from retrieval.helpers import collapse_by_title
from eval.evaluation import score_retrieval

In [2]:
CORPUS_PATH = ROOT / "data" / "corpus" / "pooled_corpus.parquet"
GOLD_DIR = ROOT / "results" / "gold"

# pid -> title, for collapsing duplicates and scoring
corpus = pd.read_parquet(CORPUS_PATH, columns=["pid", "title"])
pid_to_title = dict(zip(corpus.pid, corpus.title))

# qid -> question type (bridge or comparison)
hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")
train_types = hotpot_ds["train"].select_columns(["id", "type"]).to_pandas()
qid_to_type = dict(zip(train_types.id, train_types.type))

In [ ]:
RUNS_DIR = ROOT / "data" / "runs"
SLICES = ["pilot", "tune"]
RRF_K = 60 # default -> tunable
KEEP_K = 50

for slice_name in SLICES:
    # load bm25 runs
    with open(RUNS_DIR / f"bm25_{slice_name}.json") as f:
        bm25_run = json.load(f)

    # load Contriever runs
    with open(RUNS_DIR / f"contriever_{slice_name}.json") as f:
        contriever_run = json.load(f)

    assert bm25_run.keys() == contriever_run.keys()

    rrf_run = {}
    # score pids
    for qid in bm25_run:
        scores = defaultdict(float)

        for rank, pid in enumerate(bm25_run[qid], start=1):
            scores[pid] += 1 / (RRF_K + rank)
            
        for rank, pid in enumerate(contriever_run[qid], start=1):
            scores[pid] += 1 / (RRF_K + rank)

        # sort pids by score
        merged = sorted(scores, key=lambda p: (-scores[p], p))
        
        # collapse by title (each list may keep a different whitespace version) and keep 50
        rrf_run[qid] = collapse_by_title(merged, pid_to_title, keep=KEEP_K)


    # checks
    assert list(rrf_run) == list(bm25_run)
    assert all(len(r) == len(set(r)) <= KEEP_K for r in rrf_run.values())
    assert all(len({pid_to_title[p] for p in r}) == len(r) for r in rrf_run.values())
    
    # save run
    with open(RUNS_DIR / f"rrf_{slice_name}.json", "w") as f:
        json.dump(rrf_run, f)   


### Scoring

In [4]:
for slice_name in SLICES:
    with open(GOLD_DIR / f"{slice_name}_gold.json") as f:
        gold = json.load(f)
    with open(RUNS_DIR / f"rrf_{slice_name}.json") as f:
        rrf_run = json.load(f)
    
    _, summary = score_retrieval(rrf_run, gold, qid_to_type, pid_to_title)
    print(f"\n=== RRF (BM25 + Contriever), {slice_name} ===")
    print(pd.DataFrame(summary).T.round(2).to_string())


=== RRF (BM25 + Contriever), pilot ===
                n  both@2  any@2  mean_gold@2  both@5   any@5  mean_gold@5  both@10  any@10  mean_gold@10  both@20  any@20  mean_gold@20  both@50  any@50  mean_gold@50
overall     200.0   26.50  86.50         1.13    43.0   93.50         1.36    53.50    96.0          1.50    58.00   97.50          1.56    66.50   99.00          1.66
bridge      160.0   18.75  83.75         1.02    32.5   91.88         1.24    43.12    95.0          1.38    48.75   96.88          1.46    58.13   98.75          1.57
comparison   40.0   57.50  97.50         1.55    85.0  100.00         1.85    95.00   100.0          1.95    95.00  100.00          1.95   100.00  100.00          2.00

=== RRF (BM25 + Contriever), tune ===
                 n  both@2  any@2  mean_gold@2  both@5  any@5  mean_gold@5  both@10  any@10  mean_gold@10  both@20  any@20  mean_gold@20  both@50  any@50  mean_gold@50
overall     1000.0   24.80  87.90         1.13    45.4  93.70         1.39     56